In [1]:
import pandas as pd
psalm_verses = pd.read_csv("data/cleaned_psalm_verses.csv")
psalm_verses

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,tradition,text,psalm_num,verse_num,verse
0,Orthodox,Bible,1,1,Blessed is the man Who walks not in the counse...
1,Orthodox,Bible,1,2,But his will is in the law of the Lord And in ...
2,Orthodox,Bible,1,3,He shall be like a tree Planted by streams of ...
3,Orthodox,Bible,1,4,Not so are the ungodly not so But they are lik...
4,Orthodox,Bible,1,5,Therefore the ungodly shall not rise in the ju...
...,...,...,...,...,...
4937,Orthodox,Psalter,150,1,Praise God in His holy ones; praise Him in the...
4938,Orthodox,Psalter,150,2,Praise Him for His mighty acts; praise Him acc...
4939,Orthodox,Psalter,150,3,Praise Him with the sound of the trumpet; prai...
4940,Orthodox,Psalter,150,4,Praise Him with the timbrel and dance; praise ...


In [2]:
psalms = pd.read_csv("data/grouped_psalm.csv")
psalms

,Unnamed: 0,tradition,text,psalm_num,verse,cleaned_verse
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...
...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ..."


In [3]:
TAG_MAP = {
    # NOUNS
    'NN': 'common_noun',
    'NNS': 'common_noun_plural',
    'NNP': 'proper_noun',
    'NNPS': 'proper_noun_plural',

    # VERBS
    'VB': 'verb_base',
    'VBD': 'verb_past',
    'VBG': 'verb_gerund',
    'VBN': 'verb_participle',
    'VBP': 'verb_present',
    'VBZ': 'verb_3sg',
    'MD': 'modal',

    # ADJECTIVES
    'JJ': 'adjective',
    'JJR': 'comparative_adj',
    'JJS': 'superlative_adj',

    # ADVERBS
    'RB': 'adverb',
    'RBR': 'comparative_adv',
    'RBS': 'superlative_adv',
    'WRB': 'wh_adverb',

    # PRONOUNS
    'PRP': 'personal_pronoun',
    'PRP$': 'possessive_pronoun',
    'WP': 'wh_pronoun',
    'WP$': 'possessive_wh_pronoun',

    # DETERMINERS
    'DT': 'determiner',
    'WDT': 'wh_determiner',
    'PDT': 'predeterminer',

    # PREPOSITIONS / CONJUNCTIONS
    'IN': 'preposition',
    'CC': 'coordinating_conjunction',

    # PARTICLES
    'RP': 'particle',
    'TO': 'infinitive_to',

    # NUMBERS
    'CD': 'cardinal_number',

    # EXISTENTIALS
    'EX': 'existential_there',

    # INTERJECTIONS
    'UH': 'interjection',

    # FOREIGN WORDS
    'FW': 'foreign_word',

    # LIST MARKERS
    'LS': 'list_marker',

    # POSSESSIVE ENDINGS
    'POS': 'possessive_ending',

    # SYMBOLS
    '$': 'currency_symbol',
    '#': 'hash_symbol',
    'SYM': 'symbol',

    # PUNCTUATION
    '.': 'sentence_end',
    ',': 'comma',
    ':': 'colon_semicolon',
    '(': 'left_paren',
    ')': 'right_paren',
    '``': 'open_quote',
    "''": 'close_quote'
}

In [4]:
import nltk
from nltk.tokenize import word_tokenize


def tag_word(word):
    token = word_tokenize(word)
    # Get the raw tag (e.g., 'NN' or 'JJ')
    raw_tag = nltk.pos_tag(token)[0][1]
    
    # Return the mapped name, or 'other' if it isn't a noun or adjective
    return TAG_MAP.get(raw_tag, 'other')

# Examples
print(tag_word("apple"))  # Output: noun
print(tag_word("quick"))  # Output: adjective
print(tag_word("run"))    # Output: other

common_noun
common_noun
verb_base


# Correspondence Analysis of Part-of-Speech Distributions

Correspondence Analysis (CA) is a multivariate statistical technique used to explore relationships between categorical variables. In this study, CA is applied to part-of-speech (POS) frequencies extracted from the Bible and Psalter translations of the Psalms.

The analysis begins by counting the occurrences of major POS categories (e.g., nouns, verbs, adjectives, adverbs, and pronouns) within each Psalm. These frequencies are then organized into a contingency table, where rows represent individual Psalms (or topics) and columns represent POS categories.

CA projects both Psalms and POS categories into a lower-dimensional space, allowing patterns in grammatical style to be visualized and interpreted. Psalms located near specific POS categories exhibit a stronger association with those grammatical features.

The primary goal is to determine whether the Bible and Psalter translations differ in their grammatical profiles. In particular, the analysis investigates whether one translation tends toward a more nominal style (characterized by nouns, adjectives, and determiners) or a more verbal style (characterized by verbs, pronouns, and adverbs).

By reducing complex POS frequency data into a small number of interpretable dimensions, Correspondence Analysis provides a quantitative method for examining stylistic differences between translations and identifying broader linguistic patterns across the Psalms.

In [5]:
ca_df = pd.DataFrame(index=[0], columns=['Psalm', 'Translation'])

# target_psalm = psalms['verse'].iloc[0]

In [6]:
from collections import Counter

def tag_text(psalm, num, trans):
    counts = Counter()

    for word in psalm.split():
        pos = tag_word(word)
        counts[pos] += 1

    row = {
        'Psalm': num,
        'Translation': trans,
        **counts
    }

    return row


In [7]:
rows = []

for _, row in psalms.iterrows():
    rows.append(
        tag_text(
            row['verse'],
            row['psalm_num'],
            row['text']
        )
    )
    

ca_df = pd.DataFrame(rows).fillna(0)

ca_df.head()

,Psalm,Translation,verb_participle,verb_3sg,determiner,common_noun,wh_pronoun,common_noun_plural,adverb,preposition,...,verb_past,superlative_adj,proper_noun,existential_there,cardinal_number,comparative_adj,comparative_adv,possessive_wh_pronoun,interjection,sentence_end
0,1,Bible,2.0,3.0,24,27,1.0,11,12.0,22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Bible,5.0,0.0,21,52,2.0,15,5.0,26,...,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Bible,2.0,3.0,14,39,6.0,7,5.0,19,...,1.0,0.0,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,4,Bible,4.0,0.0,8,48,3.0,8,3.0,23,...,4.0,0.0,4.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,5,Bible,2.0,3.0,22,74,4.0,8,6.0,39,...,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
